### Concatenate

#### Zadanie 1 - łączenie raportów z wielu miesięcy


Dane przychodzą z 3 systemów sprzedażowych
Każdy system może powtórzyć request - powstają duplikaty eventów *błędna architektura API, trzeba je usunąć

ZADANIE:

1. Połącz wszystkie źródła danych.
2. Dodaj kolumnę 'zrodlo'.
3. Sprawdź liczbę rekordów przed czyszczeniem.
4. Usuń duplikaty, definiując rekord jako:
   (timestamp, klient, produkt, wartosc)
5. Sprawdź liczbę rekordów po czyszczeniu.
6. Wyświetl wynik.

In [3]:
import pandas as pd

online = pd.DataFrame({
    'timestamp': ['2024-01-01 10:00', '2024-01-01 10:05', '2024-01-01 10:10'],
    'klient': ['Jan', 'Anna', 'Piotr'],
    'produkt': ['Laptop', 'Mysz', 'Klawiatura'],
    'wartosc': [4000, 80, 250]
})

store = pd.DataFrame({
    'timestamp': ['2024-01-01 10:10', '2024-01-01 10:15', '2024-01-01 10:20'],
    'klient': ['Piotr', 'Adam', 'Kasia'],
    'produkt': ['Klawiatura', 'Monitor', 'Laptop'],
    'wartosc': [250, 1200, 4200]
})

marketplace = pd.DataFrame({
    'timestamp': ['2024-01-01 10:20', '2024-01-01 10:25', '2024-01-01 10:25'],
    'klient': ['Kasia', 'Tomek', 'Tomek'],  # retry API → duplikat eventu
    'produkt': ['Laptop', 'Słuchawki', 'Słuchawki'],
    'wartosc': [4200, 300, 300]
})

marketplace_invalid = pd.concat([marketplace, marketplace], ignore_index=True)

display(marketplace_invalid)

,timestamp,klient,produkt,wartosc
0,2024-01-01 10:20,Kasia,Laptop,4200
1,2024-01-01 10:25,Tomek,Słuchawki,300
2,2024-01-01 10:25,Tomek,Słuchawki,300
3,2024-01-01 10:20,Kasia,Laptop,4200
4,2024-01-01 10:25,Tomek,Słuchawki,300
5,2024-01-01 10:25,Tomek,Słuchawki,300


In [6]:
final_df = (
    pd.concat([
        online,
        store,
        marketplace_invalid
    ], ignore_index=True)
    # .sort_values('timestamp')
    .drop_duplicates(
        subset=['timestamp', 'klient', 'produkt', 'wartosc']
    )
)

final_df

,timestamp,klient,produkt,wartosc
0,2024-01-01 10:00,Jan,Laptop,4000
1,2024-01-01 10:05,Anna,Mysz,80
2,2024-01-01 10:10,Piotr,Klawiatura,250
4,2024-01-01 10:15,Adam,Monitor,1200
5,2024-01-01 10:20,Kasia,Laptop,4200
7,2024-01-01 10:25,Tomek,Słuchawki,300



---
#### Zadanie 2 - concat z różnymi kolumnami i diagnostyka braków

Trzy systemy eksportują dane o pracownikach, ale każdy ma nieco inne kolumny. 

Połącz je w jeden DataFrame (outer i inner), zbadaj które kolumny brakują z których systemów, 

uzupełnij braki sensownymi wartościami domyślnymi i dodaj kolumnę 'zrodlo'.

In [7]:
import pandas as pd

system_hr = pd.DataFrame({
    'emp_id': [1, 2, 3],
    'imie': ['Anna', 'Bartek', 'Celina'],
    'dzial': ['IT', 'HR', 'IT'],
    'etat': [1.0, 0.5, 1.0]
})

system_place = pd.DataFrame({
    'emp_id': [2, 3, 4],
    'imie': ['Bartek', 'Celina', 'Dawid'],
    'wynagrodzenie': [6000, 5500, 7000],
    'waluta': ['PLN', 'PLN', 'PLN']
})

system_oceny = pd.DataFrame({
    'emp_id': [1, 4, 5],
    'imie': ['Anna', 'Dawid', 'Ewa'],
    'ocena_roczna': [4.5, 3.8, 4.2],
    'czy_senior': [True, False, True]
})

In [8]:
df_hr = system_hr.assign(source='HR')
df_pl = system_place.assign(source='place')
df_sys = system_oceny.assign(source='oceny')

# outer join
outer = pd.concat([df_hr, df_pl, df_sys], ignore_index=True, join='outer')

# innej join
inner = pd.concat([df_hr, df_pl, df_sys], ignore_index=True, join='inner')

display('Ramka outer:')
display(outer)

display('Ramka inner:')
display(inner)

'Ramka outer:'

,emp_id,imie,dzial,etat,source,wynagrodzenie,waluta,ocena_roczna,czy_senior
0,1,Anna,IT,1.0,HR,NaN,NaN,NaN,NaN
1,2,Bartek,HR,0.5,HR,NaN,NaN,NaN,NaN
2,3,Celina,IT,1.0,HR,NaN,NaN,NaN,NaN
3,2,Bartek,NaN,NaN,place,6000.0,PLN,NaN,NaN
4,3,Celina,NaN,NaN,place,5500.0,PLN,NaN,NaN
5,4,Dawid,NaN,NaN,place,7000.0,PLN,NaN,NaN
6,1,Anna,NaN,NaN,oceny,NaN,NaN,4.5,True
7,4,Dawid,NaN,NaN,oceny,NaN,NaN,3.8,False
8,5,Ewa,NaN,NaN,oceny,NaN,NaN,4.2,True


'Ramka inner:'

,emp_id,imie,source
0,1,Anna,HR
1,2,Bartek,HR
2,3,Celina,HR
3,2,Bartek,place
4,3,Celina,place
5,4,Dawid,place
6,1,Anna,oceny
7,4,Dawid,oceny
8,5,Ewa,oceny


In [9]:
display(
    outer.groupby('source').apply(
        lambda x: x.isna().sum(),
        include_groups=False
    )
)

,emp_id,imie,dzial,etat,wynagrodzenie,waluta,ocena_roczna,czy_senior
source,,,,,,,,
HR,0,0,0,0,3,3,3,3
oceny,0,0,3,3,3,3,0,0
place,0,0,3,3,0,0,3,3


In [10]:
# uzupełnianie braków:

outer_filled = outer.copy()

outer_filled['wynagrodzenie'] = outer_filled['wynagrodzenie'].fillna(0)
outer_filled['waluta'] = outer_filled['waluta'].fillna('PLN')
outer_filled['etat'] = outer_filled['etat'].fillna(1.0)

outer_filled

,emp_id,imie,dzial,etat,source,wynagrodzenie,waluta,ocena_roczna,czy_senior
0,1,Anna,IT,1.0,HR,0.0,PLN,NaN,NaN
1,2,Bartek,HR,0.5,HR,0.0,PLN,NaN,NaN
2,3,Celina,IT,1.0,HR,0.0,PLN,NaN,NaN
3,2,Bartek,NaN,1.0,place,6000.0,PLN,NaN,NaN
4,3,Celina,NaN,1.0,place,5500.0,PLN,NaN,NaN
5,4,Dawid,NaN,1.0,place,7000.0,PLN,NaN,NaN
6,1,Anna,NaN,1.0,oceny,0.0,PLN,4.5,True
7,4,Dawid,NaN,1.0,oceny,0.0,PLN,3.8,False
8,5,Ewa,NaN,1.0,oceny,0.0,PLN,4.2,True
